# Regressão Linear

## Importação de bibliotecas

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, train_test_split

## Importação de dataset

In [2]:
data = pd.read_csv("/content/carne-brasileira-exportada.csv", encoding='utf-8', sep=',')
data.head()

,ano,trimestre,tipo_carne,producao_cabecas_anual_mi,abate_cabecas,peso_carcaca_ton,exportacao_usd,media_cambio_usdbrl,custo_milho_rs,custo_soja_rs,preco_medio_ton_usd
0,2014,1,bovino,212.34,8.366,1.951,1.345,2.34,22.57,62.01,4.403
1,2014,2,bovino,212.34,8.516,2.006,1.384,2.22,23.59,61.42,4.729
2,2014,3,bovino,212.34,8.456,2.036,1.547,2.28,19.97,56.50,4.874
3,2014,4,bovino,212.34,8.525,2.058,1.519,2.59,21.46,56.73,4.858
4,2014,1,suino,37.93,8.686,0.747,0.260,2.34,22.57,62.01,2.811


## Treinamento do modelo

### Seleção de features

In [3]:
features = [
    "ano",
    "trimestre",
    "producao_cabecas_anual_mi",
    "abate_cabecas",
    "peso_carcaca_ton",
    "media_cambio_usdbrl",
    "custo_milho_rs",
    "custo_soja_rs",
    "preco_medio_ton_usd",
]

target = "exportacao_usd"

### Separação, treinamento e avaliação

In [ ]:
for carne in ["bovino", "suino", "frango"]:
    df = data[data["tipo_carne"] == carne].copy()
    df = df.sort_values(["ano", "trimestre"]).reset_index(drop=True)

    X = df[features]
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False
    )

    param_grid = {
        "fit_intercept": [True, False],
        "copy_X": [True, False],
        "positive": [False, True],
    }

    grid_search = GridSearchCV(
        LinearRegression(),
        param_grid,
        scoring="neg_mean_squared_error",
        cv=3,
        n_jobs=-1,
        verbose=0,
    )
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    print(f"--- {carne} ---")
    print("Melhores parâmetros:", grid_search.best_params_)
    print("Coeficientes:", best_model.coef_)
    print("Intercept:", best_model.intercept_)
    print("MSE:", mean_squared_error(y_test, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print("R²:", r2_score(y_test, y_pred))
    print()

--- bovino ---
Melhores parâmetros: {'copy_X': True, 'fit_intercept': True, 'positive': True}
Coeficientes: [7.66583055e-02 0.00000000e+00 4.50388738e-04 0.00000000e+00
 1.76505753e+00 0.00000000e+00 1.72316536e-03 9.85515214e-04
 5.39160595e-01]
Intercept: -159.18419802839475
MSE: 0.14074138755411625
R²: 0.09812480503894216

--- suino ---
Melhores parâmetros: {'copy_X': True, 'fit_intercept': True, 'positive': True}
Coeficientes: [0.         0.01150051 0.         0.         0.49034425 0.03919741
 0.00100602 0.         0.14987364]
Intercept: -0.6736139286705914
MSE: 0.005208880102291525
R²: 0.274079886642556

--- frango ---
Melhores parâmetros: {'copy_X': True, 'fit_intercept': False, 'positive': True}
Coeficientes: [0.00000000e+00 2.34628330e-02 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 2.40925393e-04 1.68278962e-03
 8.58400144e-01]
Intercept: 0.0
MSE: 0.11808149473938598
R²: -3.726139817493853

